# NB07 - Dataset Distillation & Coreset Selection
## Background
Dataset distillation (Wang et al., 2018) asks: can we compress a large dataset into a tiny synthetic one that trains models just as well? The original paper showed that 10 synthetic MNIST images can represent 60,000 real ones for certain tasks. This is distinct from coreset selection, which selects real examples — dataset distillation synthesizes new data points.

Practical coreset methods: k-center (minimize maximum distance to nearest center), Forgetting events (Toneva et al., 2019, tracks which examples are forgotten during training), and influence-based pruning (remove examples with the lowest influence on validation loss).

## What You'll Learn
- k-center coreset: greedy farthest-point traversal
- Herding: class prototype-based selection
- Forgetting events: select examples with the most transitions from correct to incorrect
- Dataset pruning: 1000 samples down to 100 while preserving accuracy

## Why This Matters
At billion-sample LLM pretraining scale, dataset pruning can reduce training cost 3-10x with minimal accuracy loss. DSIR (Xie et al., 2023) and other data selection methods for LLMs are direct applications of these coreset principles.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Tuple

np.random.seed(42)
n_classes, n_features = 4, 6
N_LARGE = 1000
N_SMALL = 100

class_means = np.random.randn(n_classes, n_features) * 2
X_large, y_large = [], []
for c in range(n_classes):
    X_large.append(class_means[c] + np.random.randn(N_LARGE//n_classes, n_features))
    y_large.extend([c]*(N_LARGE//n_classes))
X_large, y_large = np.vstack(X_large), np.array(y_large)

# Test set (not used for selection)
X_test, y_test = [], []
for c in range(n_classes):
    X_test.append(class_means[c] + np.random.randn(50, n_features))
    y_test.extend([c]*50)
X_test, y_test = np.vstack(X_test), np.array(y_test)

print(f'Large dataset: {N_LARGE} samples -> Target coreset: {N_SMALL} samples')

# ── Coreset selection methods ──────────────────────────────────────────────
def k_center_greedy(X: np.ndarray, n_select: int, seed: int = 0) -> np.ndarray:
    """k-center: iteratively select the point farthest from all selected."""
    np.random.seed(seed)
    selected = [np.random.randint(len(X))]  # Start with random point
    for _ in range(n_select - 1):
        X_sel = X[selected]
        # Minimum distance from each point to selected set
        dists = np.array([min(np.linalg.norm(X[i] - xs) for xs in X_sel)
                          for i in range(len(X))])
        dists[selected] = -1  # Exclude already selected
        selected.append(int(np.argmax(dists)))
    return np.array(selected)

def herding_selection(X: np.ndarray, y: np.ndarray, n_select: int) -> np.ndarray:
    """Herding: class-by-class selection to approximate class mean."""
    selected = []
    n_per_class = n_select // len(np.unique(y))
    for c in np.unique(y):
        class_mask = y == c
        X_c = X[class_mask]
        class_indices = np.where(class_mask)[0]
        mu = X_c.mean(axis=0)
        selected_c = []
        running_mean = np.zeros(X.shape[1])
        for k in range(n_per_class):
            remaining = [i for i in range(len(X_c)) if i not in selected_c]
            if not remaining: break
            # Select point that minimizes distance of running mean to class mean
            best = min(remaining, key=lambda i:
                       np.linalg.norm((running_mean + X_c[i]) / (k+1) - mu))
            selected_c.append(best)
            running_mean += X_c[best]
        selected.extend([class_indices[i] for i in selected_c])
    return np.array(selected)

def random_selection(n_total: int, n_select: int, seed: int = 0) -> np.ndarray:
    np.random.seed(seed)
    return np.random.choice(n_total, n_select, replace=False)

print('Coreset methods implemented: k-center, herding, random')

In [ ]:
# ── Train and evaluate on coresets ────────────────────────────────────────
class SoftmaxClf:
    def __init__(self, seed=0):
        np.random.seed(seed)
        self.W = np.random.randn(n_features, n_classes) * 0.1
        self.b = np.zeros(n_classes)

    def softmax(self, X):
        z = X @ self.W + self.b
        z -= z.max(axis=1, keepdims=True)
        e = np.exp(z); return e / e.sum(axis=1, keepdims=True)

    def fit(self, X, y, n_epochs=200, lr=0.1):
        for _ in range(n_epochs):
            probs = self.softmax(X)
            n = len(y)
            dz = probs.copy(); dz[np.arange(n), y] -= 1; dz /= n
            self.W -= lr * X.T @ dz
            self.b -= lr * dz.sum(axis=0)

    def accuracy(self, X, y): return np.mean(np.argmax(self.softmax(X), axis=1) == y)

# Full dataset baseline
clf_full = SoftmaxClf(seed=42)
clf_full.fit(X_large, y_large)
acc_full = clf_full.accuracy(X_test, y_test)
print(f'Full dataset ({N_LARGE} samples): test acc = {acc_full:.3f}')

methods = {}

# Random coreset
rand_idx = random_selection(N_LARGE, N_SMALL)
clf_rand = SoftmaxClf(seed=42)
clf_rand.fit(X_large[rand_idx], y_large[rand_idx])
methods['Random'] = clf_rand.accuracy(X_test, y_test)

# k-center coreset
kcenter_idx = k_center_greedy(X_large, N_SMALL)
clf_kc = SoftmaxClf(seed=42)
clf_kc.fit(X_large[kcenter_idx], y_large[kcenter_idx])
methods['k-Center'] = clf_kc.accuracy(X_test, y_test)

# Herding coreset
herd_idx = herding_selection(X_large, y_large, N_SMALL)
clf_herd = SoftmaxClf(seed=42)
clf_herd.fit(X_large[herd_idx], y_large[herd_idx])
methods['Herding'] = clf_herd.accuracy(X_test, y_test)

print(f'\nCoreset results ({N_SMALL} samples = {100*N_SMALL/N_LARGE:.0f}% of data):')
for method, acc in methods.items():
    drop = (acc_full - acc) * 100
    print(f'  {method:10s}: acc = {acc:.3f} ({drop:+.1f}% vs full)')

In [ ]:
# ── Visualize coreset coverage ─────────────────────────────────────────────
from sklearn.decomposition import PCA
try:
    pca = PCA(n_components=2).fit(X_large)
    X_2d = pca.transform(X_large)
except ImportError:
    # Fallback: use first 2 features
    X_2d = X_large[:, :2]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
cmap = plt.cm.tab10

for ax, (name, idx) in zip(axes, [
    ('Random', rand_idx),
    ('k-Center', kcenter_idx),
    ('Herding', herd_idx),
]):
    ax.scatter(X_2d[:, 0], X_2d[:, 1], c=y_large, cmap=cmap, alpha=0.15, s=10)
    ax.scatter(X_2d[idx, 0], X_2d[idx, 1], c=y_large[idx], cmap=cmap,
               s=60, edgecolors='black', linewidths=0.8)
    acc = methods.get(name, 0)
    ax.set_title(f'{name} (acc={acc:.3f})')
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
    ax.grid(True, alpha=0.3)

plt.suptitle(f'Coreset Coverage: {N_SMALL}/{N_LARGE} samples selected')
plt.tight_layout()
plt.savefig(f'{BASE}/s29_07_coreset.png', dpi=100, bbox_inches='tight')
plt.show()
print('k-center maximizes spatial coverage; herding preserves class structure.')